# 03 · Baseline comparison

Runs or loads the four approaches on the same claims:

| name | pipeline |
|---|---|
| A1 conventional RAG | claim → SEC retrieval → LLM → verdict |
| A2 RAG + numeric verification | A1 → deterministic XBRL check → verdict |
| A3 contradiction-aware | A2 → challenge questions → counter-evidence retrieval → reconciliation |
| SR self-reflection | A2 → self-critique → revision (no new retrieval) |

**`RUN_MODE` (default `load`) never calls a paid API.**

- `load`: analyse the newest complete real run (or, if none exists, run a MOCK demo)
- `mock`: run the whole pipeline with the keyword mock LLM (no credentials; not results)
- `offline_cache`: run using cached model responses only
- `live`: paid Claude API calls; set `APPROVAL_PHRASE` to the exact phrase the cost plan prints

Every run is written to `results/runs/<run_id>/` and locked (read-only + SHA256SUMS) once complete.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
CONFIG = "configs/main_comparison.json"
SPLIT = "development"          # "gold" only for the final held-out evaluation
CLAIM_IDS = None               # e.g. ["DEV-AAPL-01", "DEV-MSFT-04"]
APPROACHES = None              # None = all approaches in the config
RUN_MODE = "load"              # load | mock | offline_cache | live
RUN_ID = None                  # load a specific run
RESUME_RUN_ID = None           # continue an interrupted run (offline_cache / live)
APPROVAL_PHRASE = ""           # live only: paste the confirmation phrase printed below

In [ ]:
from engine.config import load_config, config_hash, thaw
from engine.runner import select_claims, plan_run, run_experiment, load_run, verify_run_integrity
from engine.llm import approve_paid_run
from engine.notebook import banner, pick_run, describe_run
cfg = load_config(CONFIG)
print(cfg["description"])
pd.DataFrame([{"approach": n, "kind": a["kind"], "config_hash": config_hash(a), "numeric": a["numeric"]["enabled"],
               "contradiction": a["contradiction"]["enabled"], "reflection": a["reflection"]["enabled"],
               "model": a["llm"]["model"], "temperature": a["llm"]["temperature"], "max_output_tokens": a["llm"]["max_output_tokens"],
               "top_k": a["retrieval"]["top_k"]} for n, a in cfg["approaches"].items()])

In [ ]:
print("fairness problems:", cfg["fairness_problems"] or "none")
claims, info = select_claims(cfg, split=SPLIT, claim_ids=CLAIM_IDS, allow_gold=(SPLIT == "gold"))
plan = plan_run(cfg, claims, APPROACHES)
display(pd.DataFrame(plan["approaches"]))
print(f"upper bound: {plan['max_llm_calls']} calls, ${plan['max_cost_usd']:.2f}  (typical ≈ ${plan['typical_cost_usd']:.2f})")
print("To approve a live run, set APPROVAL_PHRASE to exactly:\n  ", plan["confirmation_phrase"])

In [ ]:
if RUN_MODE == "live":
    approval = approve_paid_run(plan, APPROVAL_PHRASE)   # raises unless the phrase matches exactly
    run = load_run(run_experiment(cfg, claims, info, llm_mode="live", approval=approval, approaches=APPROACHES,
                                  resume_run_id=RESUME_RUN_ID))
elif RUN_MODE == "offline_cache":
    run = load_run(run_experiment(cfg, claims, info, llm_mode="offline", approaches=APPROACHES, resume_run_id=RESUME_RUN_ID))
elif RUN_MODE == "mock":
    run = load_run(run_experiment(cfg, claims, info, llm_mode="mock", approaches=APPROACHES, progress=lambda s: None))
else:
    run = pick_run(cfg["experiment_id"], RUN_ID)
    if run is None:
        banner("No stored run for this experiment yet; running the MOCK pipeline so the analysis below has data. "
               "Mock labels are keyword heuristics, not results.", "info")
        run = load_run(run_experiment(cfg, claims, info, llm_mode="mock", approaches=APPROACHES, progress=lambda s: None))
describe_run(run)

## Predictions side by side

In [ ]:
preds = pd.DataFrame(run["predictions"])
side = preds.pivot_table(index=["claim_id", "claim_category"], columns="approach", values="predicted_label", aggfunc="first")
side

## Initial → final verdicts for the second-look approaches

In [ ]:
two = preds[preds.approach_kind.isin(["contradiction_aware", "self_reflection"])]
two[["approach", "claim_id", "claim_category", "initial_label", "predicted_label", "verdict_changed", "stop_reason"]]

## How often approaches agree

In [ ]:
names = list(side.columns)
agree = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        agree.loc[a, b] = (side[a] == side[b]).mean()
agree.round(2)

## Run health: stop reasons, errors, citations

In [ ]:
health = preds.groupby("approach").agg(n=("claim_id", "count"), errors=("error", lambda s: s.notna().sum()),
                                       unsupported=("unsupported_conclusion", "sum"),
                                       invalid_citations=("invalid_citations", lambda s: sum(len(x or []) for x in s)),
                                       avg_llm_calls=("llm_calls", "mean"), avg_retrieval_queries=("retrieval_queries", "mean"))
display(health)
display(preds.groupby(["approach", "stop_reason"]).size().unstack(fill_value=0))

## Stored artifacts

In [ ]:
print(run["run_dir"])
for root, _, files in os.walk(run["run_dir"]):
    depth = root[len(run["run_dir"]):].count(os.sep)
    if depth <= 1:
        print(" ", os.path.relpath(root, run["run_dir"]), f"({len(files)} files)")
print("integrity:", verify_run_integrity(run["run_dir"]) or "all files match SHA256SUMS")